# Laboratorio di Machine Learning per l’Analisi Finanziaria — Università Federico II
## Lezione 01 — Il churn come problema di classificazione: formulazione e analisi esplorativa (LIVE CODING)

**Autori:** Enrico Huber, Pietro Soglia  \\n
**Email:** enricohuber5@gmail.com, pietro.soglia@gmail.com  \\n
**Ultimo aggiornamento:** 2026-03-02

Nota: questa versione contiene **gap intenzionali** per il live coding.

## 1. Obiettivi di apprendimento

(Compilare/raffinare a lezione)
- (TODO) Tradurre churn in classificazione (feature/target/unità)
- (TODO) Riconoscere tipi di variabili (numeriche/categoriche/binarie)
- (TODO) EDA **guidata**: qualità del dato, distribuzioni, sbilanciamento, segmenti
- (TODO) Individuare rischi (leakage) e implicazioni su metriche/soglia

## 2. Setup

### Strumenti (cenni)
- GenAI / Code Assistants: scaffolding, debug, refactor (sempre verificare).
- Git & GitHub: versionamento e collaborazione (cenni).

### Contratto di input/output (artifact-based)
Questo notebook legge:
- `data/archive.zip` (contiene `Customer-Churn-Records.csv`)

Questo notebook scrive (sezione 3–4):
- `outputs/data/lesson_01_raw.parquet` (se disponibile) **oppure** `outputs/data/lesson_01_raw.csv` (fallback)
- `outputs/figures/lesson_01_*.png` (figure EDA con prefisso `lesson_01_`)
- `outputs/config/lesson_01_eda_summary.json` (riassunto EDA)

In [ ]:
from __future__ import annotations

from pathlib import Path
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Reproducibilità (usata quando ci sono componenti stocastici)
SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    'figure.figsize': (8, 4),
    'axes.grid': True,
})

def find_project_root(start: Path) -> Path:
    """Cerca la root del repo risalendo finché trova README.md e requirements.txt."""
    current = start.resolve()
    for parent in [current, *current.parents]:
        if (parent / 'README.md').exists() and (parent / 'requirements.txt').exists():
            return parent
    # fallback: cwd
    return current

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / 'data'
ARCHIVE_PATH = DATA_DIR / 'archive.zip'

OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
OUT_DATA_DIR = OUTPUTS_DIR / 'data'
OUT_CONFIG_DIR = OUTPUTS_DIR / 'config'
OUT_FIGURES_DIR = OUTPUTS_DIR / 'figures'
OUT_MODELS_DIR = OUTPUTS_DIR / 'models'
OUT_PRED_DIR = OUTPUTS_DIR / 'predictions'
OUT_SUB_DIR = OUTPUTS_DIR / 'submissions'

for d in [OUT_DATA_DIR, OUT_CONFIG_DIR, OUT_FIGURES_DIR, OUT_MODELS_DIR, OUT_PRED_DIR, OUT_SUB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RAW_PARQUET_PATH = OUT_DATA_DIR / 'lesson_01_raw.parquet'
TARGET_DIST_FIG_PATH = OUT_FIGURES_DIR / 'lesson_01_target_distribution.png'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('ARCHIVE_PATH:', ARCHIVE_PATH)
print('RAW_PARQUET_PATH:', RAW_PARQUET_PATH)

## 3. Data loading

(Live coding) Carichiamo il CSV dall’archivio zip e ispezioniamo schema e missingness.

In [ ]:
if not ARCHIVE_PATH.exists():
    raise FileNotFoundError(f"Non trovo {ARCHIVE_PATH}. Assicurati di avere `data/archive.zip` nel repository.")

# TODO: implementare load_csv_from_zip(...)
CSV_MEMBER = 'Customer-Churn-Records.csv'

# def load_csv_from_zip(archive_path: Path, member_name: str) -> pd.DataFrame:
#     ...

# df_raw = load_csv_from_zip(ARCHIVE_PATH, CSV_MEMBER)
# print('Shape:', df_raw.shape)
# display(df_raw.head())
# display(df_raw.dtypes)

In [ ]:
# TODO: missingness overview + salvataggio artifact

# missing = (df_raw.isna().mean() * 100).sort_values(ascending=False)
# missing = missing[missing > 0]
# if len(missing) == 0:
#     print('Nessun valore mancante (0% missing)')
# else:
#     display(missing.to_frame(name='missing_%'))

# try:
#     df_raw.to_parquet(RAW_PARQUET_PATH, index=False)
#     print('Salvato:', RAW_PARQUET_PATH)
# except Exception as e:
#     fallback_path = RAW_PARQUET_PATH.with_suffix('.csv')
#     df_raw.to_csv(fallback_path, index=False)
#     print('Parquet non disponibile (es. manca pyarrow).')
#     print('Salvato fallback CSV:', fallback_path)
#     print('Dettagli errore:', repr(e))

## 4. Exploratory analysis (EDA)

(Live coding) Target distribution, statistiche descrittive, feature principali e imbalance.

### 4.1 Qualità del dataset (sanity checks)

(Live coding) Duplicati, colonne ID-like, cardinalità categoriche.

In [ ]:
# TODO: sanity checks come nello student notebook
## TARGET_COL_NAME = 'Exited'
## n_rows, n_cols = df_raw.shape
## print('Righe:', n_rows, '| Colonne:', n_cols)

## n_dup_rows = int(df_raw.duplicated().sum())
## print('Duplicati di riga:', n_dup_rows)

## if 'CustomerId' in df_raw.columns:
##     n_dup_customer = int(df_raw['CustomerId'].duplicated().sum())
##     print('CustomerId duplicati:', n_dup_customer)

## nunique = df_raw.nunique(dropna=False).sort_values(ascending=False)
## constant_cols = nunique[nunique == 1].index.tolist()
## if len(constant_cols) > 0:
##     print('Colonne costanti:', constant_cols)
## else:
##     print('Nessuna colonna costante')

## print('Nota: variabili continue (float) possono avere molti valori unici senza essere ID.')
## id_like_cols = []
## for col, n_u in nunique.items():
##     if col == TARGET_COL_NAME:
##         continue
##     if n_u >= 0.95 * n_rows:
##         is_discrete = pd.api.types.is_integer_dtype(df_raw[col]) or pd.api.types.is_object_dtype(df_raw[col])
##         if is_discrete:
##             id_like_cols.append(col)
## print('Colonne ID-like (>=95% unici, discrete):', id_like_cols)

## cat_cols_all = df_raw.select_dtypes(include=['object', 'string']).columns.tolist()
## if len(cat_cols_all) > 0:
##     display(nunique[cat_cols_all].sort_values(ascending=False).to_frame('n_unique'))

**Osservazioni (qualità dati, dal run completo dello student notebook)**

- Il dataset ha **10.000 righe** e **18 colonne**; non risultano duplicati di righe né duplicati di `CustomerId`.
- Le colonne **ID-like** sono `RowNumber` e `CustomerId`: in genere vanno escluse dalle feature.
- `Surname` ha cardinalità alta (**2932**): attenzione a come codificarla (o valutare di rimuoverla).
- `Geography` (3) e `Gender` (2) sono categoriche a bassa cardinalità, adatte a one-hot encoding.

In [ ]:
# TODO: target distribution (come nello student notebook)
# TARGET_COL = 'Exited'
# if TARGET_COL not in df_raw.columns:
#     raise KeyError(f"Colonna target '{TARGET_COL}' non trovata. Colonne: {list(df_raw.columns)}")

# target_counts = df_raw[TARGET_COL].value_counts(dropna=False).sort_index()
# target_perc = target_counts / len(df_raw) * 100
# display(pd.DataFrame({'count': target_counts, 'perc_%': target_perc.round(2)}))

# fig, ax = plt.subplots()
# ax.bar(target_counts.index.astype(str), target_counts.values)
# ax.set_title('Distribuzione del target (Exited)')
# ax.set_xlabel('Classe')
# ax.set_ylabel('Conteggio')
# fig.tight_layout()
# fig.savefig(TARGET_DIST_FIG_PATH, dpi=150)
# plt.show()
# print('Salvata figura:', TARGET_DIST_FIG_PATH)

**Interpretazione (target e sbilanciamento, dal run completo dello student notebook)**

- `Exited=1` è **20,38%** e `Exited=0` è **79,62%**.
- Accuracy da sola può essere fuorviante (baseline “sempre 0” ≈ 79,62%).
- Nelle lezioni successive useremo ROC-AUC e discuteremo soglia/costi; per lo split conviene stratificare sul target.

In [ ]:
# TODO: statistiche descrittive + confronto per classe (come nello student notebook)
# display(df_raw.describe(include='number').T)

# num_cols = df_raw.select_dtypes(include='number').columns.tolist()
# num_cols = [c for c in num_cols if c != TARGET_COL]
# group_means = df_raw.groupby(TARGET_COL)[num_cols].mean(numeric_only=True)
# display(group_means.T.sort_values(by=1, ascending=False).head(15))

**Spunti dai confronti per classe (medie numeriche, dal run completo)**

- `Age`: churner più anziani (≈ **44,84** vs **37,41**).
- `Balance`: più alto per churner (≈ **91.109** vs **72.743**).
- `IsActiveMember`: più basso per churner (≈ **0,361** vs **0,555**).
- `Complain`: quasi deterministico (≈ **0,998** vs ≈ **0,001**) → possibile leakage.

In [ ]:
# TODO: distribuzione Age per classe (come nello student notebook)
# if 'Age' in df_raw.columns:
#     fig, ax = plt.subplots()
#     for cls, color in [(0, 'tab:blue'), (1, 'tab:orange')]:
#         subset = df_raw.loc[df_raw[TARGET_COL] == cls, 'Age']
#         ax.hist(subset, bins=20, alpha=0.6, label=f'{TARGET_COL}={cls}', color=color)
#     ax.set_title('Distribuzione Age per classe')
#     ax.set_xlabel('Age')
#     ax.set_ylabel('Frequenza')
#     ax.legend()
#     fig.tight_layout()
#     plt.show()

**Interpretazione (Age)**

- La distribuzione di `Age` tende a essere più alta per `Exited=1` (coerente con la media).
- Le distribuzioni si sovrappongono: `Age` da sola non basta, ma è un buon segnale da combinare.
- Ponte verso la segmentazione: churn rate per fasce (quantili/decili).

### 4.2 Distribuzioni numeriche per classe

(Live coding) Aggiungiamo altre feature numeriche (Balance, CreditScore, EstimatedSalary), boxplot e segmentazione per quantili.

In [ ]:
# TODO: helper per istogrammi per classe (come nello student notebook)
## def plot_hist_by_target(df: pd.DataFrame, col: str, target_col: str, bins: int, out_path: Path) -> None:
##     """Istogramma sovrapposto per una feature numerica, separando per classe target."""
##     fig, ax = plt.subplots()
##     for cls, color in [(0, 'tab:blue'), (1, 'tab:orange')]:
##         subset = df.loc[df[target_col] == cls, col].dropna()
##         ax.hist(subset, bins=bins, alpha=0.6, label=f'{target_col}={cls}', color=color)
##     ax.set_title(f'Distribuzione {col} per classe')
##     ax.set_xlabel(col)
##     ax.set_ylabel('Frequenza')
##     ax.legend()
##     fig.tight_layout()
##     fig.savefig(out_path, dpi=150)
##     plt.show()
##     print('Salvata figura:', out_path)

# TODO: elenco colonne numeriche da plottare
## NUM_PLOT_COLS = [c for c in ['Balance', 'CreditScore', 'EstimatedSalary'] if c in df_raw.columns]
## print('Colonne numeriche plottate:', NUM_PLOT_COLS)

In [ ]:
# TODO: Plot 1: Balance (come nello student notebook)
## if 'Balance' in df_raw.columns:
##     plot_hist_by_target(
##         df_raw,
##         col='Balance',
##         target_col=TARGET_COL,
##         bins=30,
##         out_path=OUT_FIGURES_DIR / 'lesson_01_balance_hist_by_class.png',
##     )

In [ ]:
# TODO: Plot 2: CreditScore (come nello student notebook)
## if 'CreditScore' in df_raw.columns:
##     plot_hist_by_target(
##         df_raw,
##         col='CreditScore',
##         target_col=TARGET_COL,
##         bins=30,
##         out_path=OUT_FIGURES_DIR / 'lesson_01_credit_score_hist_by_class.png',
##     )

In [ ]:
# TODO: Plot 3: EstimatedSalary (come nello student notebook)
## if 'EstimatedSalary' in df_raw.columns:
##     plot_hist_by_target(
##         df_raw,
##         col='EstimatedSalary',
##         target_col=TARGET_COL,
##         bins=30,
##         out_path=OUT_FIGURES_DIR / 'lesson_01_estimated_salary_hist_by_class.png',
##     )

**Interpretazione (feature numeriche: Balance, CreditScore, EstimatedSalary)**

- Dal run completo: `Balance` differenzia abbastanza (media ≈ 91k per churner vs ≈ 73k per non churner).
- `CreditScore` ha differenze più piccole (segnale più debole).
- `EstimatedSalary` è molto simile tra classi (segnale debole).

Messaggio: singole feature raramente separano perfettamente; serve combinazione di feature e modelli capaci di interazioni.

In [ ]:
# TODO: Boxplot Age e Balance per classe (come nello student notebook)
## cols_for_box = [c for c in ['Age', 'Balance'] if c in df_raw.columns]
## if len(cols_for_box) > 0:
##     fig, axes = plt.subplots(1, len(cols_for_box), figsize=(5 * len(cols_for_box), 4), sharey=False)
##     if len(cols_for_box) == 1:
##         axes = [axes]
##     for ax, col in zip(axes, cols_for_box):
##         data0 = df_raw.loc[df_raw[TARGET_COL] == 0, col].dropna()
##         data1 = df_raw.loc[df_raw[TARGET_COL] == 1, col].dropna()
##         ax.boxplot([data0, data1], labels=[f'{TARGET_COL}=0', f'{TARGET_COL}=1'])
##         ax.set_title(f'Boxplot {col} per classe')
##         ax.set_xlabel('Classe')
##         ax.set_ylabel(col)
##     fig.tight_layout()
##     out_path = OUT_FIGURES_DIR / 'lesson_01_boxplots_age_balance.png'
##     fig.savefig(out_path, dpi=150)
##     plt.show()
##     print('Salvata figura:', out_path)

**Interpretazione (boxplot)**

- Il boxplot evidenzia mediana, dispersione e outlier; è utile per confronti robusti tra classi.
- Dal run completo: `Age` e `Balance` tendono ad avere valori centrali più alti per `Exited=1`.

In [ ]:
# TODO: Segmentazione: churn rate per quantili (come nello student notebook)
## def churn_rate_by_quantile(df: pd.DataFrame, col: str, target_col: str, q: int = 10) -> pd.DataFrame:
##     x = df[col]
##     bins = pd.qcut(x, q=q, duplicates='drop')
##     out = (
##         df.assign(_bin=bins)
##           .groupby('_bin', observed=True)[target_col]
##           .agg(['mean', 'count'])
##           .rename(columns={'mean': 'churn_rate', 'count': 'n'})
##     )
##     return out.reset_index().rename(columns={'_bin': f'{col}_bin'})

## cols_for_bins = [c for c in ['Age', 'Balance', 'CreditScore'] if c in df_raw.columns]
## for col in cols_for_bins:
##     display(churn_rate_by_quantile(df_raw, col=col, target_col=TARGET_COL, q=10).head())

**Interpretazione (segmentazione per quantili)**

- Dal run completo: `Age` mostra un aumento del churn nelle fasce più alte; `Balance` tende ad aumentare il churn passando a fasce più alte; `CreditScore` è più piatto.
- Queste tabelle sono un ponte naturale verso non-linearità e modelli ad albero/boosting.

In [ ]:
# TODO: feature categoriche: tabella P(Exited=1 | categoria) (come nello student notebook)
# cat_candidates = ['Geography', 'Gender', 'Card Type']
# cat_cols = [c for c in cat_candidates if c in df_raw.columns]
# for col in cat_cols:
#     ct = pd.crosstab(df_raw[col], df_raw[TARGET_COL], normalize='index')
#     display(ct.style.format('{:.2%}').set_caption(f'P({TARGET_COL}=1 | {col}) per categoria'))

**Interpretazione (categorie: $P(Exited=1 \mid categoria)$)**

- Queste tabelle mettono in evidenza differenze tra gruppi in modo immediato.
- Controllare sempre anche la numerosità: categorie rare possono essere instabili.
- Nel run completo vedremo differenze marcate in `Geography` e `Gender`.

### 4.3 Tasso di churn per categorie (e variabili binarie)

(Live coding) Usiamo barplot del churn rate per Geography/Gender e binarie come IsActiveMember/Complain.

In [ ]:
# TODO: helper churn_rate_by_category + plot_churn_rate_bar (come nello student notebook)
## def churn_rate_by_category(df: pd.DataFrame, col: str, target_col: str) -> pd.DataFrame:
##     ...

## def plot_churn_rate_bar(df_rate: pd.DataFrame, col: str, out_path: Path, top_n: int = 15) -> None:
##     ...

In [ ]:
# TODO: Plot churn rate per Geography (come nello student notebook)
## if 'Geography' in df_raw.columns:
##     df_geo = churn_rate_by_category(df_raw, col='Geography', target_col=TARGET_COL)
##     display(df_geo)
##     plot_churn_rate_bar(df_geo, col='Geography', out_path=OUT_FIGURES_DIR / 'lesson_01_churn_rate_by_geography.png', top_n=20)

**Interpretazione (Geography, dal run completo)**

- `Germany` ~**32,44%** vs `Spain` ~**16,67%** e `France` ~**16,17%**.
- Ottimo esempio di segmentazione e di possibili fattori latenti (contesto diverso).

In [ ]:
# TODO: Plot churn rate per Gender (come nello student notebook)
## if 'Gender' in df_raw.columns:
##     df_gender = churn_rate_by_category(df_raw, col='Gender', target_col=TARGET_COL)
##     display(df_gender)
##     plot_churn_rate_bar(df_gender, col='Gender', out_path=OUT_FIGURES_DIR / 'lesson_01_churn_rate_by_gender.png', top_n=10)

**Interpretazione (Gender, dal run completo)**

- `Female` ~**25,07%** vs `Male` ~**16,47%**.
- Spunto: verificare se l’effetto resta dopo aver condizionato su `Geography`/`Age` (interazioni).

In [ ]:
# TODO: Plot churn rate per IsActiveMember (come nello student notebook)
## if 'IsActiveMember' in df_raw.columns:
##     df_rate = churn_rate_by_category(df_raw, col='IsActiveMember', target_col=TARGET_COL)
##     display(df_rate)
##     plot_churn_rate_bar(df_rate, col='IsActiveMember', out_path=OUT_FIGURES_DIR / 'lesson_01_churn_rate_by_isactivemember.png', top_n=10)

**Interpretazione (IsActiveMember, dal run completo)**

- `IsActiveMember=0`: churn ~**26,87%**; `IsActiveMember=1`: churn ~**14,27%**.
- Lettura naturale: l’attività/engagement è associata a minor churn; esplorare interazioni con altri segmenti.

In [ ]:
# TODO: Plot churn rate per Complain (come nello student notebook)
## if 'Complain' in df_raw.columns:
##     df_rate = churn_rate_by_category(df_raw, col='Complain', target_col=TARGET_COL)
##     display(df_rate)
##     plot_churn_rate_bar(df_rate, col='Complain', out_path=OUT_FIGURES_DIR / 'lesson_01_churn_rate_by_complain.png', top_n=10)

**Interpretazione critica (Complain = possibile leakage, dal run completo)**

- `Complain=1`: churn ~**99,51%**; `Complain=0`: churn ~**0,05%**.
- Questo è un segnale fortissimo di possibile variabile post-evento: discutere timeline e fare esperimenti “con/senza”.

### 4.4 Correlazioni e relazioni bivariate (solo esplorative)

(Live coding) Correlazioni con target, heatmap (top feature), scatter Age vs Balance.

In [ ]:
# TODO: correlazioni con target + heatmap (come nello student notebook)
## num_cols_all = df_raw.select_dtypes(include='number').columns.tolist()
## if TARGET_COL in num_cols_all:
##     num_cols_all = [c for c in num_cols_all if c != TARGET_COL]

## id_like_to_drop = [c for c in ['RowNumber', 'CustomerId'] if c in df_raw.columns]
## num_cols_for_corr = [c for c in num_cols_all if c not in id_like_to_drop]

## corr_with_target = df_raw[num_cols_for_corr + [TARGET_COL]].corr(numeric_only=True)[TARGET_COL].drop(TARGET_COL)
## corr_table = corr_with_target.reindex(corr_with_target.abs().sort_values(ascending=False).index)
## display(corr_table.to_frame('corr_with_target'))

## top_corr_cols = corr_table.abs().head(12).index.tolist()
## corr_mat = df_raw[top_corr_cols].corr(numeric_only=True)

## fig, ax = plt.subplots(figsize=(8, 6))
## im = ax.imshow(corr_mat.values, vmin=-1, vmax=1, cmap='coolwarm')
## ax.set_xticks(range(len(top_corr_cols)))
## ax.set_yticks(range(len(top_corr_cols)))
## ax.set_xticklabels(top_corr_cols, rotation=45, ha='right')
## ax.set_yticklabels(top_corr_cols)
## fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
## ax.set_title('Correlazioni (feature numeriche selezionate)')
## fig.tight_layout()
## out_path = OUT_FIGURES_DIR / 'lesson_01_correlation_heatmap.png'
## fig.savefig(out_path, dpi=150)
## plt.show()
## print('Salvata figura:', out_path)

**Interpretazione (correlazioni, dal run completo)**

- `Complain` è estremamente correlata con `Exited` (≈ 0,996): molto probabilmente leakage.
- Tra le feature “plausibili” emergono `Age` (≈ 0,285), `IsActiveMember` (≈ -0,156), `Balance` (≈ 0,119).
- Pearson misura soprattutto relazioni lineari: correlazione bassa non implica “inutile”.

In [ ]:
# TODO: scatter Age vs Balance (come nello student notebook)
## if 'Age' in df_raw.columns and 'Balance' in df_raw.columns:
##     fig, ax = plt.subplots(figsize=(7, 5))
##     for cls, color in [(0, 'tab:blue'), (1, 'tab:orange')]:
##         sub = df_raw.loc[df_raw[TARGET_COL] == cls, ['Age', 'Balance']].dropna()
##         ax.scatter(sub['Age'], sub['Balance'], s=10, alpha=0.25, c=color, label=f'{TARGET_COL}={cls}')
##     ax.set_title('Age vs Balance (color: classe)')
##     ax.set_xlabel('Age')
##     ax.set_ylabel('Balance')
##     ax.legend()
##     fig.tight_layout()
##     out_path = OUT_FIGURES_DIR / 'lesson_01_age_vs_balance_scatter.png'
##     fig.savefig(out_path, dpi=150)
##     plt.show()
##     print('Salvata figura:', out_path)

**Interpretazione (scatter Age vs Balance)**

- È una vista bivariata: aiuta a capire se le classi si separano in una regione del piano.
- Nel run completo si nota una massa di punti a `Balance=0`: spesso conviene aggiungere un indicatore `Balance>0`.
- Anche qui la separazione non è netta: è un promemoria che servirà un modello (e non una regola a mano).

In [ ]:
# TODO: salvare un riassunto EDA in outputs/config (come nello student notebook)
## import json
## eda_summary = {
##     'n_rows': int(n_rows),
##     'n_cols': int(n_cols),
##     'target_col': TARGET_COL,
##     'churn_rate': float(df_raw[TARGET_COL].mean()),
##     'id_like_cols': id_like_cols,
## }
## summary_path = OUT_CONFIG_DIR / 'lesson_01_eda_summary.json'
## with open(summary_path, 'w', encoding='utf-8') as f:
##     json.dump(eda_summary, f, ensure_ascii=False, indent=2)
## print('Salvato riassunto EDA:', summary_path)

**Artifact (riassunto EDA su disco)**

Nel materiale completo salviamo un riassunto in `outputs/config/lesson_01_eda_summary.json`. Serve per riusare numeri chiave (churn rate, segmenti top, correlazioni) nelle lezioni successive senza dipendere dallo stato del kernel.

**Mini-sintesi EDA (messaggi chiave, dal run completo)**

- Churn rate complessivo: **20,38%** → sbilanciamento ~80/20.
- Segmenti forti: `Geography` (Germany ~32,44%), `Gender` (Female ~25,07%), `IsActiveMember` (0 ~26,87% vs 1 ~14,27%).
- Segnali numerici: `Age` e `Balance` (differenze visibili, ma sovrapposizione ampia).
- Variabile sospetta: `Complain` (quasi deterministica) → discutere leakage.
- Colonne da trattare con cura: ID-like (`RowNumber`, `CustomerId`) e `Surname` (alta cardinalità).

### Discussione

Domande guida (da discutere in aula):
- Il target è sbilanciato (~80/20): quale metrica usereste e perché?
- Quali feature sembrano più diverse tra `Exited=0` e `Exited=1` (es. `Age`, `Balance`, `IsActiveMember`)?
- `Complain` sembra quasi deterministica: quali indizi vi fanno pensare a leakage? Che controllo “di processo” fareste sui dati?
- Il churn rate cambia molto per `Geography` (Germany > Spain/France): quali ipotesi plausibili possono spiegare la differenza?
- Come gestireste `Surname` (alta cardinalità) in una pipeline reale?
- Se doveste proporre 2–3 azioni di retention, su quali segmenti le concentrereste e perché?

## 5. Preprocessing
(Placeholder) Anticipazione: decisioni su colonne, missing, encoding, scaling.

In [ ]:
# TODO (Lesson 02): pipeline preprocessing
pass

## 6. Modeling
(Placeholder) Baseline e confronto modelli in Lesson 03.

In [ ]:
# TODO (Lesson 03): baseline
pass

## 7. Evaluation
(Placeholder) ROC-AUC e metriche secondarie in Lesson 03.

In [ ]:
# TODO (Lesson 03): evaluation
pass

## 8. Interpretation
(Placeholder) Feature importance in Lesson 04.

In [ ]:
# TODO (Lesson 04): interpretabilità
pass

## 9. Esercizi
- [easy] Percentuale churn per Geography
- [easy] Boxplot Balance per classe
- [medium] 3 feature candidate e motivazione
- [hard] proposta strategia di split

## 10. Riepilogo
(Compilare a fine lezione)
- (TODO) punti chiave e pitfalls